# LitePhospho — E4 only: MusiteDeep 2020 released models (CPU, no GPU quota)

Same notebook as the reviewer experiments, with E1–E3 switched off. **Accelerator: None, Internet: On**, add the *bioinformatics-thesis-project* dataset, then **Save & Run All**. The full output of every shell command is saved to `exp_results/E4_log.txt` — download that file and `SUMMARY.md` afterwards.

---

# (Original description) LitePhospho — reviewer-response experiments (single Kaggle notebook)

Runs the four GPU experiments a TCBB reviewer is likely to ask for, on the **released
benchmark split** (`data/litephospho_split.csv` from github.com/karimulislambd/litephospho):

| Exp | Question it answers | Runs |
|---|---|---|
| **E1** | Is the leakage inflation (random − homology) different between architectures, beyond seed noise? Is LitePhospho's lead over DeepPhos\*/MusiteDeep\* stable? | 3 models × 2 splits × `SEEDS` |
| **E2** | Does *flatten instead of global pooling* really matter? (controlled ablation, incl. a capacity-matched pooled variant) | 3 variants × `SEEDS` |
| **E3** | How far below a *stronger* protein language model is LitePhospho? Frozen ESM2-35M and ESM2-650M with whole-protein context, plus ESM2-35M **fine-tuned** on the same 31-residue window | 2 frozen × `SEEDS` heads + 1 fine-tune |
| **E4** | How does a **published tool, as released** (MusiteDeep 2020 webserver models) score on our test proteins? | 1 inference run (CPU) |

**All deep models in E1/E2 use one identical trainer** (AdamW lr 3e-4, wd 1e-3, batch 512,
early stopping on validation loss, patience 12) — this removes the "unequal tuning" concern.

### How to run on Kaggle
1. *Settings → Accelerator:* **GPU T4 x2** (or P100). *Settings → Internet:* **On** (needed for
   ESM2 weights, the split file and MusiteDeep).
2. *Add data:* your datasets **bioinformatics-thesis-project** (EPSD + UniProt) — the notebook finds
   `Homo sapiens.txt` and `uniprotkb.fasta` anywhere under `/kaggle/input`.
3. **Save Version → Save & Run All (Commit)**. Expected total ≈ 6–8 h on a T4 (estimates below).
4. **Resume after a timeout:** every finished run is written to `/kaggle/working/exp_results/`
   immediately. If a session dies, add this notebook's previous *output* as input data
   (*Add data → Notebook Output*) and run again — finished runs are detected and skipped.
5. Download `/kaggle/working/exp_results/` (especially `SUMMARY.md` and `summary.json`) and send
   them back to update the paper.

In [ ]:
# ===================== CONFIG — edit here only =====================
SEEDS            = [1, 2, 3]      # training seeds (use [1,2,3,4,5] if time allows)
RUN_E1_SEEDS     = False          # multi-seed LitePhospho / DeepPhos* / MusiteDeep*, random vs homology
RUN_E2_ABLATION  = False          # flatten vs global pooling
RUN_E3_ESM       = False          # frozen ESM2-35M + ESM2-650M (whole-protein context)
RUN_E3_FINETUNE  = False          # fine-tune ESM2-35M end-to-end on the 31-residue window
RUN_E4_MUSITE    = True           # MusiteDeep 2020 released models (CPU, separate Python 3.6 env)
USE_AMP          = False          # mixed precision on GPU (≈2x faster on T4)
TIME_LIMIT_H     = 11.3           # stop starting new runs after this (Kaggle hard limit is 12 h)
SPLIT_URL = 'https://raw.githubusercontent.com/karimulislambd/litephospho/main/data/litephospho_split.csv'
# ====================================================================
import time; T0 = time.time()
def elapsed_h(): return (time.time() - T0) / 3600
def time_ok(est_min=0):
    ok = elapsed_h() + est_min / 60 < TIME_LIMIT_H
    if not ok: print(f'  !! time budget reached ({elapsed_h():.2f} h) — skipping; resume in a new session')
    return ok

In [ ]:
# ===================== environment, paths, results store =====================
import os, re, json, glob, copy, shutil, subprocess, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

assert os.path.exists('/kaggle/input'), 'Run this on Kaggle (attach the datasets first).'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| GPUs:', torch.cuda.device_count(), '| torch', torch.__version__)

def find_one(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input — attach the bioinformatics-thesis-project dataset'
    return Path(hits[0])
EPSD_FILE = find_one('Homo sapiens.txt'); UNIPROT_FILE = find_one('uniprotkb.fasta')
print('EPSD   :', EPSD_FILE); print('UniProt:', UNIPROT_FILE)

OUT = Path('/kaggle/working/exp_results'); (OUT / 'scores').mkdir(parents=True, exist_ok=True)
RES_FILE = OUT / 'results.json'
RESULTS = json.load(open(RES_FILE)) if RES_FILE.exists() else {}
# resume: merge results from a previous version's output attached as input data
for prev in glob.glob('/kaggle/input/**/exp_results/results.json', recursive=True):
    old = json.load(open(prev)); new = {k: v for k, v in old.items() if k not in RESULTS}
    RESULTS.update(new)
    for f in glob.glob(str(Path(prev).parent / 'scores' / '*.npy')):
        dst = OUT / 'scores' / Path(f).name
        if not dst.exists(): shutil.copy(f, dst)
    print(f'resumed {len(new)} finished runs from {prev}')
def save_result(key, metrics, scores=None):
    RESULTS[key] = metrics
    if scores is not None: np.save(OUT / 'scores' / (key.replace('|', '__') + '.npy'), scores.astype(np.float16))
    json.dump(RESULTS, open(RES_FILE, 'w'), indent=1)
def done(key):
    if key in RESULTS: print(f'  skip (done): {key}'); return True
    return False
print(f'{len(RESULTS)} finished runs on record')

In [ ]:
# ===================== definitions: seeding, models, fast trainer =====================
WINDOW, HALF = 31, 15
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWYX'
PAD_IDX = AMINO_ACIDS.index('X'); NAA = len(AMINO_ACIDS)
LUT = np.full(256, PAD_IDX, np.int64)
for i, a in enumerate(AMINO_ACIDS): LUT[ord(a)] = i

def seed_all(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def encode(windows):   # vectorised: list[str] -> int64 [n, 31]
    b = np.frombuffer(''.join(windows).upper().encode('ascii'), np.uint8).reshape(len(windows), WINDOW)
    return LUT[b]

# ---- models: identical to the canonical pipeline (litephospho_pipeline / leakage experiment) ----
class LitePhospho(nn.Module):
    def __init__(self, vocab_size=NAA, embed_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.c3 = nn.Conv1d(embed_dim, 64, 3, padding=1)
        self.c5 = nn.Conv1d(embed_dim, 64, 5, padding=2)
        self.c7 = nn.Conv1d(embed_dim, 64, 7, padding=3)
        self.bn1 = nn.BatchNorm1d(192)
        self.conv2 = nn.Conv1d(192, 64, 3, padding=1); self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 32, 1); self.bn3 = nn.BatchNorm1d(32)
        self.fc1 = nn.Linear(32 * WINDOW, 256); self.bn4 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 64); self.fc3 = nn.Linear(64, 1); self.drop = nn.Dropout(dropout)
    def _head(self, h):
        x = self.drop(F.relu(self.bn4(self.fc1(h))))
        x = self.drop(F.relu(self.fc2(x)))
        return self.fc3(x).squeeze(1)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        h = torch.cat([F.relu(self.c3(emb)), F.relu(self.c5(emb)), F.relu(self.c7(emb))], dim=1)
        h = self.drop(self.bn1(h))
        h = self.drop(self.bn2(F.relu(self.conv2(h))))
        h = self.drop(self.bn3(F.relu(self.conv3(h))))
        return self._head(h.flatten(1))

class LitePhosphoPooled(LitePhospho):
    """E2 ablation: identical trunk, but global pooling over the window instead of flatten.
    mode='gap'  global average pool (32-d)          - position discarded, fewer fc1 params
    mode='gmp'  global max pool (32-d)
    mode='gap_wide' conv3 widened to 992 channels, then GAP -> 992-d: same fc1 size as flatten,
                so any gap to LitePhospho is due to discarding position, not to capacity."""
    def __init__(self, mode='gap'):
        super().__init__()
        self.mode = mode
        if mode == 'gap_wide':
            self.conv3 = nn.Conv1d(64, 32 * WINDOW, 1); self.bn3 = nn.BatchNorm1d(32 * WINDOW)
        else:
            self.fc1 = nn.Linear(32, 256)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        h = torch.cat([F.relu(self.c3(emb)), F.relu(self.c5(emb)), F.relu(self.c7(emb))], dim=1)
        h = self.drop(self.bn1(h))
        h = self.drop(self.bn2(F.relu(self.conv2(h))))
        h = self.drop(self.bn3(F.relu(self.conv3(h))))
        h = h.max(dim=2).values if self.mode == 'gmp' else h.mean(dim=2)
        return self._head(h)

class DeepPhosNet(nn.Module):
    def __init__(self, vocab_size=NAA, embed_dim=64, growth=32, kernels=(1, 3, 5, 7, 9), dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.blocks = nn.ModuleList()
        for k in kernels:
            layers, in_c = nn.ModuleList(), embed_dim
            for _ in range(3):
                layers.append(nn.Conv1d(in_c, growth, k, padding=k // 2)); in_c += growth
            self.blocks.append(layers)
        ch = len(kernels) * (embed_dim + 3 * growth)
        self.reduce = nn.Conv1d(ch, 32, 1); self.bn = nn.BatchNorm1d(32)
        self.fc1 = nn.Linear(32 * WINDOW, 128); self.fc2 = nn.Linear(128, 1); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2); outs = []
        for layers in self.blocks:
            h = emb
            for conv in layers: h = torch.cat([h, F.relu(conv(h))], dim=1)
            outs.append(h)
        z = self.drop(F.relu(self.bn(self.reduce(torch.cat(outs, dim=1))))).flatten(1)
        return self.fc2(self.drop(F.relu(self.fc1(z)))).squeeze(1)

class MusiteDeepNet(nn.Module):
    def __init__(self, vocab_size=NAA, embed_dim=64, ch=128, heads=4, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.conv1 = nn.Conv1d(embed_dim, ch, 5, padding=2)
        self.conv2 = nn.Conv1d(ch, ch, 3, padding=1); self.bn = nn.BatchNorm1d(ch)
        self.attn = nn.MultiheadAttention(ch, heads, dropout=dropout, batch_first=True)
        self.fc1 = nn.Linear(ch * WINDOW, 128); self.fc2 = nn.Linear(128, 1); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        h = self.bn(F.relu(self.conv2(F.relu(self.conv1(emb))))).transpose(1, 2)
        a, _ = self.attn(h, h, h)
        z = self.drop(a).flatten(1)
        return self.fc2(self.drop(F.relu(self.fc1(z)))).squeeze(1)

MODELS = {'LitePhospho': LitePhospho, 'DeepPhos': DeepPhosNet, 'MusiteDeep': MusiteDeepNet}
for n, C in MODELS.items(): print(f'{n:12s} {sum(p.numel() for p in C().parameters()):>9,} params')
for m in ('gap', 'gmp', 'gap_wide'):
    print(f'LP-{m:9s} {sum(p.numel() for p in LitePhosphoPooled(m).parameters()):>9,} params')

def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (p > edges[i]) & (p <= edges[i + 1])
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)

def metrics(y, s):
    return dict(ROC_AUC=float(roc_auc_score(y, s)), PR_AUC=float(average_precision_score(y, s)),
                F1=float(f1_score(y, (s >= 0.5).astype(int))), ECE=ece(y, s))

@torch.no_grad()
def predict(model, X, bs=8192):
    model.eval(); out = []
    for k in range(0, len(X), bs):
        with torch.autocast('cuda', enabled=USE_AMP and device.type == 'cuda'):
            out.append(torch.sigmoid(model(X[k:k + bs]).float()))
    return torch.cat(out).cpu().numpy()

def train_eval(factory, D, seed, epochs=80, lr=3e-4, wd=1e-3, patience=12, bs=512):
    """Same algorithm as train_eval_split() of the leakage experiment (AdamW, BCE, early stopping on
    val loss, best-val checkpoint), but with GPU-resident tensors (much faster) and optional AMP."""
    seed_all(seed); t = time.time()
    m = factory().to(device); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    crit = nn.BCEWithLogitsLoss(); scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    g = torch.Generator(device=device); g.manual_seed(seed)
    Xtr, ytr, Xva, yva = D['Xtr'], D['ytr'], D['Xva'], D['yva']; n = len(ytr)
    best, wait, best_sd, ep_used = 1e9, 0, None, 0
    for ep in range(1, epochs + 1):
        m.train(); perm = torch.randperm(n, device=device, generator=g)
        for k in range(0, n, bs):
            idx = perm[k:k + bs]
            if len(idx) < 2: continue                     # BatchNorm needs >1 sample
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=USE_AMP):
                loss = crit(m(Xtr[idx]).float(), ytr[idx])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        m.eval(); tot = 0.0
        with torch.no_grad():
            for k in range(0, len(yva), 8192):
                with torch.autocast('cuda', enabled=USE_AMP):
                    tot += crit(m(Xva[k:k + 8192]).float(), yva[k:k + 8192]).item() * len(yva[k:k + 8192])
        vl = tot / len(yva)
        if vl < best: best, wait, ep_used = vl, 0, ep; best_sd = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(best_sd)
    s = predict(m, D['Xte']); res = metrics(D['yte_np'], s)
    res.update(best_epoch=ep_used, val_loss=float(best), minutes=round((time.time() - t) / 60, 2),
               params=int(sum(p.numel() for p in m.parameters())), seed=seed)
    return res, s, m
print('definitions ready')

In [ ]:
# ===================== data: EPSD + UniProt -> verified windows (canonical pipeline) =====================
df = pd.read_csv(EPSD_FILE, sep=None, engine='python')
acc_regex = re.compile(r'^[OPQ][0-9][A-Z0-9]{3}[0-9](?:-\d+)?$|^[A-NR-Z][0-9]{5}(?:-\d+)?$', re.I)
acc_col = max(df.columns, key=lambda c: df[c].astype(str).str.strip().str.match(acc_regex).sum())
pos_col = next(c for c in df.columns if 'position' in c.lower())
res_col = next(c for c in df.columns if c.lower() in {'residue', 'aa', 'modified residue'})
epsd = df[[acc_col, res_col, pos_col]].copy(); epsd.columns = ['acc_raw', 'residue', 'pos']
epsd['acc'] = epsd['acc_raw'].astype(str).str.split('-').str[0]
epsd['pos'] = epsd['pos'].astype(int); epsd['residue'] = epsd['residue'].astype(str).str.upper()
epsd = epsd[epsd['residue'].isin(list('STY'))].drop_duplicates(['acc', 'pos'])
print('EPSD S/T/Y sites (dedup):', f'{len(epsd):,}')

seq_dict, acc, buf = {}, None, []
for line in open(UNIPROT_FILE):
    line = line.strip()
    if line.startswith('>'):
        if acc: seq_dict[acc] = ''.join(buf)
        p = line[1:].split('|'); acc = p[1] if len(p) >= 2 else p[0].split()[0]; buf = []
    else: buf.append(line)
if acc: seq_dict[acc] = ''.join(buf)
print('UniProt sequences:', f'{len(seq_dict):,}')

ek = epsd[epsd['acc'].isin(seq_dict)]
pos_records = []
for a, grp in ek.groupby('acc'):                       # groupby is sorted -> deterministic
    s = seq_dict[a]; padded = 'X' * HALF + s + 'X' * HALF
    for p, r in zip(grp['pos'].values - 1, grp['residue'].values):
        if 0 <= p < len(s) and s[p] == r:              # residue must match the EPSD annotation
            pos_records.append((a, int(p), padded[p:p + WINDOW], 1))
pos_set = {(a, p) for a, p, _, _ in pos_records}
pos_proteins = sorted({a for a, _, _, _ in pos_records})   # sorted (canonical code iterated a set)
neg_records = []
for a in pos_proteins:
    s = seq_dict[a]; padded = 'X' * HALF + s + 'X' * HALF
    for i, aa in enumerate(s):
        if aa in 'STY' and (a, i) not in pos_set:
            neg_records.append((a, i, padded[i:i + WINDOW], 0))
rng = np.random.default_rng(42)
neg_bal = [neg_records[i] for i in rng.choice(len(neg_records), len(pos_records), replace=False)]
all_records = pos_records + neg_bal
print(f'positives {len(pos_records):,} (paper 577,608) | raw negatives {len(neg_records):,} (paper 1,272,526) '
      f'| balanced {len(all_records):,} on {len(pos_proteins):,} proteins')

In [ ]:
# ===================== splits: released homology split + matched random split =====================
try:
    split_df = pd.read_csv(SPLIT_URL); print('split file: downloaded from GitHub')
except Exception as e:
    split_df = pd.read_csv(find_one('litephospho_split.csv')); print('split file: from /kaggle/input', e)
acc2split = dict(zip(split_df['accession'], split_df['split']))
B = {'train': [], 'val': [], 'test': []}
for r in all_records:
    sp = acc2split.get(r[0])
    if sp in B: B[sp].append(r)
print('homology split (released):', {k: f'{len(v):,}' for k, v in B.items()},
      '| positives:', {k: f'{np.mean([r[3] for r in v]):.1%}' for k, v in B.items()})

flat = B['train'] + B['val'] + B['test']
perm = np.random.default_rng(42).permutation(len(flat)); ntr, nva = len(B['train']), len(B['val'])
R = {'train': [flat[i] for i in perm[:ntr]], 'val': [flat[i] for i in perm[ntr:ntr + nva]],
     'test': [flat[i] for i in perm[ntr + nva:]]}

def to_tensors(S):
    D = {}
    for k, part in (('tr', 'train'), ('va', 'val'), ('te', 'test')):
        recs = S[part]
        D['X' + k] = torch.from_numpy(encode([r[2] for r in recs])).to(device)
        y = np.array([r[3] for r in recs], np.float32)
        D['y' + k] = torch.from_numpy(y).to(device)
        if k == 'te': D['yte_np'] = y; D['te_keys'] = [(r[0], r[1]) for r in recs]
    return D
SPLITS = {'homology': to_tensors(B), 'random': to_tensors(R)}
TEST_KEYS = SPLITS['homology']['te_keys']; Y_TEST = SPLITS['homology']['yte_np']
np.save(OUT / 'scores' / 'y_test_homology.npy', Y_TEST)
pd.DataFrame(TEST_KEYS, columns=['accession', 'pos0']).to_csv(OUT / 'test_windows_homology.csv', index=False)
print('tensors on', device)

In [ ]:
# ===================== E1 — multi-seed: 3 models x {homology, random} =====================
# est. minutes per run on a T4 (GPU-resident data + AMP): LitePhospho ~4, DeepPhos ~10, MusiteDeep ~8
EST = {'LitePhospho': 5, 'DeepPhos': 12, 'MusiteDeep': 10}
if RUN_E1_SEEDS:
    for seed in SEEDS:                                   # seed-major: partial runs still give whole seeds
        for name, Cls in MODELS.items():
            for split in ('homology', 'random'):
                key = f'E1|{name}|{split}|seed{seed}'
                if done(key) or not time_ok(EST[name]): continue
                res, s, _ = train_eval(Cls, SPLITS[split], seed)
                save_result(key, res, s)
                print(f'{key:38s} ROC {res["ROC_AUC"]:.4f} PR {res["PR_AUC"]:.4f} '
                      f'ep {res["best_epoch"]:>2} {res["minutes"]:.1f} min | total {elapsed_h():.2f} h')

In [ ]:
# ===================== E2 — ablation: flatten vs global pooling (homology split) =====================
if RUN_E2_ABLATION:
    for seed in SEEDS:
        for mode in ('gap', 'gmp', 'gap_wide'):
            key = f'E2|LP-{mode}|homology|seed{seed}'
            if done(key) or not time_ok(6): continue
            res, s, _ = train_eval(lambda: LitePhosphoPooled(mode), SPLITS['homology'], seed)
            save_result(key, res, s)
            print(f'{key:38s} ROC {res["ROC_AUC"]:.4f} PR {res["PR_AUC"]:.4f} params {res["params"]:,} | {elapsed_h():.2f} h')
    print('reference = E1|LitePhospho|homology|seed* (same trainer, same seeds)')

In [ ]:
# ===================== E3a — frozen ESM2 (35M, 650M) with whole-protein context =====================
# Each protein is embedded once (chunks of <=1022 residues with 50% overlap, each residue taken from
# the chunk where it is most central). Per window: mean over the 31-residue window + centre residue,
# i.e. the same pooling as the paper's ESM2-35M reference, but with full-protein context.
if RUN_E3_ESM or RUN_E3_FINETUNE:
    subprocess.run('pip install -q fair-esm', shell=True, check=True)
    import esm

class ESMHead2(nn.Module):        # identical to the paper's ESM head
    def __init__(self, d, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(256, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(64, 1))
    def forward(self, x): return self.net(x).squeeze(1)

@torch.no_grad()
def protein_reps(model, alphabet, layer, seq, maxlen=1022):
    bc = alphabet.get_batch_converter(); L = len(seq)
    if L <= maxlen: starts = [0]
    else:
        starts = sorted(set(list(range(0, L - maxlen + 1, maxlen // 2)) + [L - maxlen]))
    best = torch.full((L,), -1, dtype=torch.long); full = None
    for st in starts:
        _, _, toks = bc([('p', seq[st:st + maxlen])])
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            rep = model(toks.to(device), repr_layers=[layer])['representations'][layer][0, 1:-1].float()
        if full is None: full = torch.zeros(L, rep.shape[1], device=device)
        idx = torch.arange(st, st + rep.shape[0])
        centrality = torch.minimum(idx - st, st + rep.shape[0] - 1 - idx)
        take = centrality > best[idx]
        full[idx[take].to(device)] = rep[take.to(device)]; best[idx[take]] = centrality[take]
    return full

def esm_window_features(esm_name, layer, S):
    model, alphabet = getattr(esm.pretrained, esm_name)(); model = model.eval().to(device)
    if device.type == 'cuda': model = model.half()
    dim = model.embed_dim; feats = {}
    by_prot = {}
    for part in ('train', 'val', 'test'):
        feats[part] = np.zeros((len(S[part]), 2 * dim), np.float16)
        for i, r in enumerate(S[part]): by_prot.setdefault(r[0], []).append((part, i, r[1]))
    t = time.time()
    for n_done, (a, items) in enumerate(sorted(by_prot.items(), key=lambda kv: len(seq_dict[kv[0]]))):
        rep = protein_reps(model, alphabet, layer, seq_dict[a]); L = rep.shape[0]
        cs = torch.cat([torch.zeros(1, dim, device=device), rep.cumsum(0)])
        pos = torch.tensor([p for _, _, p in items], device=device)
        lo, hi = (pos - HALF).clamp(min=0), (pos + HALF + 1).clamp(max=L)
        f = torch.cat([(cs[hi] - cs[lo]) / (hi - lo).unsqueeze(1), rep[pos]], 1).cpu().numpy().astype(np.float16)
        for part in ('train', 'val', 'test'):
            sel = [j for j, it in enumerate(items) if it[0] == part]
            if sel: feats[part][[items[j][1] for j in sel]] = f[sel]
        if n_done % 2000 == 0: print(f'  {esm_name}: {n_done:,}/{len(by_prot):,} proteins, {(time.time()-t)/60:.1f} min')
    del model; torch.cuda.empty_cache()
    return feats, round((time.time() - t) / 60, 1)

def train_esm_head(feats, S, seed, epochs=100, patience=10):
    seed_all(seed)
    Xtr = torch.from_numpy(feats['train']).to(device); Xva = torch.from_numpy(feats['val']).to(device)
    ytr = torch.tensor([r[3] for r in S['train']], dtype=torch.float32, device=device)
    yva = torch.tensor([r[3] for r in S['val']], dtype=torch.float32, device=device)
    head = ESMHead2(Xtr.shape[1]).to(device); opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(); g = torch.Generator(device=device); g.manual_seed(seed)
    best, wait, sd = 1e9, 0, None
    for ep in range(epochs):
        head.train(); perm = torch.randperm(len(ytr), device=device, generator=g)
        for k in range(0, len(ytr), 1024):
            idx = perm[k:k + 1024]
            if len(idx) < 2: continue
            opt.zero_grad(); crit(head(Xtr[idx].float()), ytr[idx]).backward(); opt.step()
        head.eval()
        with torch.no_grad():
            vl = sum(crit(head(Xva[k:k + 8192].float()), yva[k:k + 8192]).item() * len(yva[k:k + 8192])
                     for k in range(0, len(yva), 8192)) / len(yva)
        if vl < best: best, wait, sd = vl, 0, copy.deepcopy(head.state_dict())
        else:
            wait += 1
            if wait >= patience: break
    head.load_state_dict(sd); head.eval()
    Xte = torch.from_numpy(feats['test']).to(device)
    with torch.no_grad():
        s = torch.cat([torch.sigmoid(head(Xte[k:k + 8192].float())) for k in range(0, len(Xte), 8192)]).cpu().numpy()
    del Xtr, Xva, Xte; torch.cuda.empty_cache()
    return s, sum(p.numel() for p in head.parameters())

ESM_RUNS = [('esm2_t12_35M_UR50D', 12, 35_000_000, 20), ('esm2_t33_650M_UR50D', 33, 650_000_000, 90)]
if RUN_E3_ESM:
    for esm_name, layer, n_enc, est in ESM_RUNS:
        keys = [f'E3|{esm_name}-frozen-fullctx|homology|seed{sd}' for sd in SEEDS]
        if all(k in RESULTS for k in keys): print('  skip (done):', esm_name); continue
        if not time_ok(est): continue
        feats, emb_min = esm_window_features(esm_name, layer, B)
        for k, sd in zip(keys, SEEDS):
            if k in RESULTS: continue
            t = time.time(); s, hp = train_esm_head(feats, B, sd)
            res = metrics(Y_TEST, s); res.update(seed=sd, encoder_params=n_enc, head_params=int(hp),
                                                 embed_minutes=emb_min, head_minutes=round((time.time()-t)/60, 2))
            save_result(k, res, s); print(f'{k:52s} ROC {res["ROC_AUC"]:.4f} PR {res["PR_AUC"]:.4f} | {elapsed_h():.2f} h')
        del feats; import gc; gc.collect()

In [ ]:
# ===================== E3b — fine-tune ESM2-35M end-to-end on the same 31-residue window =====================
# Input = the residues of the 31-window only (no extra context, termini not padded), i.e. exactly the
# information LitePhospho sees. Features = [centre, window-mean] of the last layer -> small MLP.
class ESMFineTune(nn.Module):
    def __init__(self, base, layer):
        super().__init__(); self.base, self.layer = base, layer
        d = base.embed_dim; self.head = nn.Sequential(nn.Linear(2 * d, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))
    def forward(self, toks, centre):
        rep = self.base(toks, repr_layers=[self.layer])['representations'][self.layer]
        mask = ((toks != self.base.padding_idx) & (toks != self.base.cls_idx) & (toks != self.base.eos_idx)).unsqueeze(-1)
        mean = (rep * mask).sum(1) / mask.sum(1)
        cen = rep[torch.arange(len(toks), device=toks.device), centre]
        return self.head(torch.cat([cen, mean], 1).float()).squeeze(1)

def window_batch(recs, bc):
    data, centre = [], []
    for a, p, _, _ in recs:
        s = seq_dict[a]; lo = max(0, p - HALF); data.append(('w', s[lo:p + HALF + 1])); centre.append(p - lo + 1)
    _, _, toks = bc(data)
    return toks.to(device), torch.tensor(centre, device=device)

def finetune_esm(esm_name, layer, S, seed, epochs=3, bs=256, lr_base=2e-5, lr_head=1e-3, patience=1):
    seed_all(seed); base, alphabet = getattr(esm.pretrained, esm_name)(); bc = alphabet.get_batch_converter()
    m = ESMFineTune(base, layer).to(device)
    opt = torch.optim.AdamW([{'params': m.base.parameters(), 'lr': lr_base},
                             {'params': m.head.parameters(), 'lr': lr_head}], weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP); crit = nn.BCEWithLogitsLoss()
    tr, va, te = S['train'], S['val'], S['test']; rng_ = np.random.default_rng(seed)
    def run_eval(recs):
        m.eval(); out = []
        with torch.no_grad():
            for k in range(0, len(recs), 1024):
                toks, c = window_batch(recs[k:k + 1024], bc)
                with torch.autocast('cuda', enabled=USE_AMP): out.append(torch.sigmoid(m(toks, c).float()).cpu())
        return torch.cat(out).numpy()
    yva = np.array([r[3] for r in va], np.float32); best, wait, sd, t = 1e9, 0, None, time.time()
    for ep in range(epochs):
        m.train(); order = rng_.permutation(len(tr))
        for k in range(0, len(order), bs):
            batch = [tr[i] for i in order[k:k + bs]]; toks, c = window_batch(batch, bc)
            y = torch.tensor([r[3] for r in batch], dtype=torch.float32, device=device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=USE_AMP): loss = crit(m(toks, c).float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        pv = np.clip(run_eval(va), 1e-7, 1 - 1e-7)
        vl = float(-np.mean(yva * np.log(pv) + (1 - yva) * np.log(1 - pv)))
        print(f'  fine-tune {esm_name} epoch {ep+1}: val loss {vl:.4f} ({(time.time()-t)/60:.1f} min)')
        if vl < best: best, wait, sd = vl, 0, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(sd); s = run_eval(te)
    res = metrics(Y_TEST, s); res.update(seed=seed, params=int(sum(p.numel() for p in m.parameters())),
                                         minutes=round((time.time() - t) / 60, 1), val_loss=best)
    del m, base; torch.cuda.empty_cache(); return res, s

if RUN_E3_FINETUNE:
    key = f'E3|esm2_t12_35M_UR50D-finetuned-window|homology|seed{SEEDS[0]}'
    if not done(key) and time_ok(80):
        res, s = finetune_esm('esm2_t12_35M_UR50D', 12, B, SEEDS[0])
        save_result(key, res, s); print(f'{key} ROC {res["ROC_AUC"]:.4f} PR {res["PR_AUC"]:.4f} | {elapsed_h():.2f} h')

In [ ]:
# ===================== E4 — MusiteDeep 2020 (released models, as published) =====================
# github.com/duolinwang/MusiteDeep_web (Wang et al., NAR 2020). Needs Python 3.6 + TensorFlow 1.12 +
# Keras 2.2.4, so it runs in an isolated micromamba env on CPU. We score every S/T/Y on the test
# proteins and evaluate on exactly the same test windows as all other models.
# CAVEAT for the paper: MusiteDeep's own training data (UniProt annotations) may include our test
# proteins or their homologues, so this comparison is, if anything, biased in MusiteDeep's favour.
def sh(cmd, timeout=None):
    """Run a shell command; the FULL stdout/stderr of every command is appended to E4_log.txt."""
    print('$', cmd[:160]); r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    with open(OUT / 'E4_log.txt', 'a', encoding='utf-8') as lg:
        lg.write(f'\n$ {cmd}\n[exit {r.returncode}]\n--- stdout ---\n{r.stdout}\n--- stderr ---\n{r.stderr}\n')
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f'exit {r.returncode}: {cmd[:80]} (see E4_log.txt)')
    return r.stdout

def parse_musite(path):
    """Parse MusiteDeep's *_results.txt (format from predict_multi_batch.py / ProtIDResult.__str__):
         Position\tResidue\tPTMscores\tCutoff=0.5        <- header (once)
         P12345                                         <- protein id line (no '>', no tab)
         17\tS\tPhosphoserine:0.734\tPhosphoserine:0.734 <- 1-based position, residue, scores, passed
       Score labels: Phosphoserine / Phosphothreonine (S/T model) and Phosphotyrosine (Y model)."""
    label = {'S': 'Phosphoserine', 'T': 'Phosphothreonine', 'Y': 'Phosphotyrosine'}
    scores, cur, mismatch = {}, None, 0
    for line in open(path):
        line = line.rstrip('\n')
        if not line.strip() or line.startswith('Position\t'): continue
        if '\t' not in line: cur = line.strip().lstrip('>').split()[0].split('|')[0]; continue
        parts = line.split('\t')
        if cur is None or len(parts) < 3 or not parts[0].strip().isdigit(): continue
        p0, aa = int(parts[0]) - 1, parts[1].strip()
        if cur not in seq_dict or p0 >= len(seq_dict[cur]) or seq_dict[cur][p0] != aa: mismatch += 1; continue
        sc = dict(kv.split(':', 1) for kv in parts[2].split(';') if ':' in kv)
        if label.get(aa) in sc: scores[(cur, p0)] = float(sc[label[aa]])
    return scores, mismatch

KEY_E4 = 'E4|MusiteDeep2020-released|homology|na'
if RUN_E4_MUSITE and not done(KEY_E4) and time_ok(90):
    try:
        W = Path('/tmp/musite'); W.mkdir(exist_ok=True); env = W / 'env'
        if not (W / 'bin/micromamba').exists():
            sh(f'cd {W} && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba')
        if not (env / 'bin/python').exists():
            sh(f'{W}/bin/micromamba create -y -q -p {env} -c conda-forge python=3.6 pip', timeout=1800)
            sh(f'{env}/bin/pip install -q tensorflow==1.12.0 keras==2.2.4 numpy==1.16.6 h5py==2.10.0 '
               f'protobuf==3.19.6 scipy==1.5.4 pandas==1.1.5 scikit-learn==0.24.2 pillow==8.4.0', timeout=1800)
        if not (W / 'MusiteDeep_web').exists():
            sh(f'cd {W} && git clone -q --depth 1 https://github.com/duolinwang/MusiteDeep_web.git')
        md = W / 'MusiteDeep_web' / 'MusiteDeep'
        test_prots = sorted({a for a, _ in TEST_KEYS})
        with open(W / 'test.fasta', 'w') as f:
            for a in test_prots: f.write(f'>{a}\n{seq_dict[a]}\n')
        # The script reloads all 20 ensemble weights for every batch of 500 sites; a larger batch
        # gives identical predictions with ~10x fewer reloads (much faster on CPU).
        pf = md / 'predict_multi_batch.py'; code_ = pf.read_bytes().decode('utf-8', 'replace')
        new_, n_ = re.subn(r'(?m)^(\s*)batch_size\s*=\s*500(\s*)$', r'\g<1>batch_size=5000\g<2>', code_)
        if n_: pf.write_bytes(new_.encode('utf-8')); print(f'patched batch_size 500 -> 5000 ({n_} line)')
        else:
            print('WARNING: batch_size line not found - running unpatched (slower, same predictions)')
            print([l for l in code_.splitlines() if 'batch_size' in l][:5])
        t = time.time()
        # -model-prefix = <model folder>/<ptm1>;<ptm2>  (folder given once, PTM names separated by ';')
        sh(f'cd {md} && CUDA_VISIBLE_DEVICES="" {env}/bin/python predict_multi_batch.py -input {W}/test.fasta '
           f'-output {W}/md_out -model-prefix "models/Phosphoserine_Phosphothreonine;Phosphotyrosine"',
           timeout=8 * 3600)
        minutes = round((time.time() - t) / 60, 1)
        outfile = W / 'md_out_results.txt'; shutil.copy(outfile, OUT / 'musitedeep2020_raw_results.txt')
        print(open(outfile).read()[:800])                 # show the format for the record
        md_score, mismatch = parse_musite(outfile)
        s = np.array([md_score.get(k, np.nan) for k in TEST_KEYS]); cover = float(np.mean(~np.isnan(s)))
        print(f'MusiteDeep 2020: scored {cover:.1%} of test windows | residue mismatches {mismatch}')
        ok = ~np.isnan(s); res = metrics(Y_TEST[ok], s[ok])
        res.update(coverage=cover, n_eval=int(ok.sum()), minutes=minutes, note='released 2020 models, CPU')
        save_result(KEY_E4, res, np.nan_to_num(s, nan=-1.0))
        print(f'{KEY_E4} ROC {res["ROC_AUC"]:.4f} PR {res["PR_AUC"]:.4f} on {ok.sum():,} windows')
    except Exception as e:
        print('E4 failed (the other experiments are unaffected):', repr(e))
        json.dump({'error': repr(e)}, open(OUT / 'E4_error.json', 'w'))

In [ ]:
# ===================== SUMMARY — tables, statistics, figure =====================
from scipy import stats
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def runs(prefix):
    return {k: v for k, v in RESULTS.items() if k.startswith(prefix)}
def load_scores(key): return np.load(OUT / 'scores' / (key.replace('|', '__') + '.npy')).astype(np.float32)
def mean_sd(v): return f'{np.mean(v):.4f} ± {np.std(v, ddof=1):.4f}' if len(v) > 1 else (f'{v[0]:.4f}' if v else 'n/a')
def ci95(v):
    if len(v) < 2: return 'n/a'
    h = stats.t.ppf(0.975, len(v) - 1) * np.std(v, ddof=1) / np.sqrt(len(v))
    return f'[{np.mean(v)-h:+.4f}, {np.mean(v)+h:+.4f}]'
def paired_boot(y, a, b, B_=1000, seed=0):
    r = np.random.default_rng(seed); n = len(y); d = []
    for _ in range(B_):
        i = r.integers(0, n, n); d.append(average_precision_score(y[i], a[i]) - average_precision_score(y[i], b[i]))
    d = np.array(d); return float(d.mean()), np.percentile(d, [2.5, 97.5]).tolist(), float(np.mean(d <= 0))

L = ['# LitePhospho reviewer experiments — summary', '',
     f'Seeds: {SEEDS} | AMP: {USE_AMP} | total runtime {elapsed_h():.2f} h | split: released litephospho_split.csv', '',
     f'Homology test set: {len(Y_TEST):,} windows ({Y_TEST.mean():.1%} positive)', '']
summary = {}

# ---- E1 ----
L += ['## E1 — multi-seed benchmark and leakage inflation', '',
      '| Model | Homology ROC | Homology PR | Random ROC | Inflation (ROC) mean ± sd | 95% CI | n |', '|---|---|---|---|---|---|---|']
infl = {}
for name in MODELS:
    hR = {r['seed']: r for k, r in runs(f'E1|{name}|homology').items()}
    rR = {r['seed']: r for k, r in runs(f'E1|{name}|random').items()}
    common = sorted(set(hR) & set(rR)); infl[name] = [rR[s]['ROC_AUC'] - hR[s]['ROC_AUC'] for s in common]
    L.append(f'| {name} | {mean_sd([r["ROC_AUC"] for r in hR.values()])} | {mean_sd([r["PR_AUC"] for r in hR.values()])} | '
             f'{mean_sd([r["ROC_AUC"] for r in rR.values()])} | {mean_sd(infl[name])} | {ci95(infl[name])} | {len(common)} |')
    summary[f'E1_{name}'] = dict(hom_roc=[r['ROC_AUC'] for r in hR.values()], hom_pr=[r['PR_AUC'] for r in hR.values()],
                                 rnd_roc=[r['ROC_AUC'] for r in rR.values()], inflation=infl[name])
L += ['', 'Welch t-tests on inflation (does architecture change inflation beyond seed noise?):']
for a, b in (('DeepPhos', 'LitePhospho'), ('MusiteDeep', 'LitePhospho'), ('MusiteDeep', 'DeepPhos')):
    if len(infl.get(a, [])) > 1 and len(infl.get(b, [])) > 1:
        t_, p_ = stats.ttest_ind(infl[a], infl[b], equal_var=False)
        L.append(f'- {a} − {b}: Δinflation {np.mean(infl[a]) - np.mean(infl[b]):+.4f}, t = {t_:.2f}, p = {p_:.4g}')
        summary[f'welch_{a}_vs_{b}'] = dict(diff=float(np.mean(infl[a]) - np.mean(infl[b])), t=float(t_), p=float(p_))
# seed-averaged scores -> paired bootstrap on the homology test set
def seed_avg(prefix):
    ks = sorted(runs(prefix)); return np.mean([load_scores(k) for k in ks], 0) if ks else None
avg = {n: seed_avg(f'E1|{n}|homology') for n in MODELS}
if all(v is not None for v in avg.values()):
    L += ['', 'Paired bootstrap on seed-averaged homology scores (ΔPR-AUC, LitePhospho − other, B = 1000):']
    for other in ('DeepPhos', 'MusiteDeep'):
        m_, ci_, p_ = paired_boot(Y_TEST, avg['LitePhospho'], avg[other])
        L.append(f'- vs {other}: {m_:+.4f} [{ci_[0]:+.4f}, {ci_[1]:+.4f}], one-sided p = {p_:.4f}')
        summary[f'boot_LP_vs_{other}'] = dict(mean=m_, ci=ci_, p=p_)

# ---- E2 ----
L += ['', '## E2 — flatten vs global pooling (homology split, same trainer and seeds)', '',
      '| Variant | Params | ROC-AUC | PR-AUC | n |', '|---|---|---|---|---|']
lp = list(runs('E1|LitePhospho|homology').values())
if lp: L.append(f'| LitePhospho (flatten) | {lp[0]["params"]:,} | {mean_sd([r["ROC_AUC"] for r in lp])} | {mean_sd([r["PR_AUC"] for r in lp])} | {len(lp)} |')
for mode in ('gap', 'gmp', 'gap_wide'):
    rr = list(runs(f'E2|LP-{mode}|').values())
    if rr:
        L.append(f'| LP-{mode} | {rr[0]["params"]:,} | {mean_sd([r["ROC_AUC"] for r in rr])} | {mean_sd([r["PR_AUC"] for r in rr])} | {len(rr)} |')
        summary[f'E2_{mode}'] = dict(roc=[r['ROC_AUC'] for r in rr], pr=[r['PR_AUC'] for r in rr])

# ---- E3 / E4 ----
L += ['', '## E3 — protein language models / E4 — published tool as released', '',
      '| Model | ROC-AUC | PR-AUC | ECE | n | note |', '|---|---|---|---|---|---|']
groups = {}
for k, r in list(runs('E3|').items()) + list(runs('E4|').items()):
    groups.setdefault(k.rsplit('|', 1)[0], []).append(r)
for g, rr in groups.items():
    note = f'coverage {rr[0]["coverage"]:.1%}' if 'coverage' in rr[0] else ''
    L.append(f'| {g.split("|")[1]} | {mean_sd([r["ROC_AUC"] for r in rr])} | {mean_sd([r["PR_AUC"] for r in rr])} | '
             f'{mean_sd([r["ECE"] for r in rr])} | {len(rr)} | {note} |')
    summary[g] = rr
if (OUT / 'E4_error.json').exists(): L.append(f'\nE4 error: {json.load(open(OUT / "E4_error.json"))}')

open(OUT / 'SUMMARY.md', 'w', encoding='utf-8').write('\n'.join(L)); json.dump(summary, open(OUT / 'summary.json', 'w'), indent=1)
print('\n'.join(L))

if all(len(infl.get(n, [])) for n in MODELS):
    plt.figure(figsize=(6.5, 4.2)); names = list(MODELS)
    mu = [np.mean(infl[n]) for n in names]; sd = [np.std(infl[n], ddof=1) if len(infl[n]) > 1 else 0 for n in names]
    plt.bar(names, mu, yerr=sd, capsize=6, color=['#e74c3c', '#557a99', '#557a99'])
    for i, n in enumerate(names): plt.scatter([i] * len(infl[n]), infl[n], color='black', s=14, zorder=3)
    plt.ylabel('ROC-AUC inflation (random − homology)'); plt.title(f'Leakage inflation, mean ± sd over {len(SEEDS)} seeds')
    plt.tight_layout(); plt.savefig(OUT / 'fig_inflation_seeds.png', dpi=300); plt.close()
print('\nSaved to', OUT, '->', sorted(p.name for p in OUT.iterdir()))